# Voice AI Meetup — 2 Hour Workshop

| Time | Topic |
|---|---|
| 0–10 min | Why Voice AI & workshop overview |
| 10–25 min | Traditional Voice Pipeline (STT → LLM → TTS) |
| 25–35 min | Why Realtime APIs & Agentic Voice Architecture |
| 35–50 min | GPT Realtime API concepts & event model |
| 50–90 min | Hands-on: browser-based voice agent (live app demo) |
| 90–110 min | Function/tool calling with the FastAPI backend |
| 110–120 min | Prompt engineering, advanced capabilities, Q&A |

This notebook carries the talking points and just enough live code to make each
concept concrete. The full end-to-end demo lives in `../backend` — we'll switch
to the browser for that partway through.

## 0. Why Voice is the Next UI

The interface layer keeps shifting:

- **Desktop → GUI**
- **Mobile → Touch**
- **ChatGPT → Text**
- **AI Agents → Voice**

Where this shows up already: customer support, restaurant ordering, medical
assistants, personal assistants, coding assistants.

> **Question for the room:** why is voice harder than chat?

(Give it a beat — answers usually surface latency, interruptions, and tone.
That's exactly what the architecture section explains.)

## 1. How Voice AI Works — the Traditional Pipeline

```
User
 ↓
Speech-to-Text   (Whisper)
 ↓
LLM              (GPT)
 ↓
Tool Calls       (Database/API)
 ↓
LLM
 ↓
Text-to-Speech
 ↓
Audio
```

Let's run this pipeline live — start to finish, timing every step.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from voice_utils import record_audio, play_audio
import time

load_dotenv()
client = OpenAI()

Record a real question from the mic — grab a volunteer from the room.

In [ ]:
record_audio("sample_question.wav", duration=5)

In [ ]:
t0 = time.time()
transcript = client.audio.transcriptions.create(model="whisper-1", file=open("sample_question.wav", "rb")).text
transcript

In [ ]:
answer = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": transcript}]).choices[0].message.content
answer

In [ ]:
client.audio.speech.create(model="tts-1", voice="alloy", input=answer, response_format="wav").stream_to_file("sample_answer.wav")
print(f"Total pipeline time: {time.time() - t0:.1f}s")
play_audio("sample_answer.wav")

Three separate API calls just to answer one question. That's the drawback in a nutshell:

- Multiple network round-trips
- Noticeable end-to-end latency
- No handling for interruptions — the user can't barge in
- Nothing streams — you wait for each stage to fully finish
- The result feels robotic, not conversational

## 2. Why Realtime APIs Exist

**Without realtime:**

```
Speak → Wait → Upload audio → Transcribe → LLM →
Generate response → Convert to speech → Download → Play
```
Latency: **2–6 seconds**

**With realtime:**

```
Mic → Streaming Audio → GPT → Streaming Audio Back
```

| Traditional Pipeline | GPT Realtime API |
|---|---|
| Separate STT, LLM, TTS services | Unified multimodal model |
| Higher latency | Low-latency streaming |
| Multiple API integrations | Single session API |
| Text-first interaction | Native audio interaction |
| Complex orchestration | Simpler architecture |

This is the biggest conceptual jump in the whole workshop: one persistent
session instead of three chained services. Everything else follows from it —
low latency, natural interruption handling, and a model that reacts to *tone*,
not just words.

## 3. Realtime API Architecture

```
Browser
 ↓ WebRTC/WebSocket
GPT Realtime API
 ↓
Audio In → Reasoning → Audio Out
```

Key pieces:

- **Session** — one long-lived connection for the whole conversation
- **Events** — every state change is a typed event on that connection
- **Audio chunks** — streamed both directions, not sent as whole files
- **Conversation state** — held server-side for the life of the session
- **Streaming responses** — text/audio arrive incrementally, not all at once

## 4. What We're Building Today

```
Browser → Mic → Realtime API → Function Calling → FastAPI
                                                      ↓
                                   get_weather / calculate / get_order_status
                                                      ↓
                                        Realtime → Speaker
```

This is the exact shape of the app in `../backend`. Time to switch over.

## → Live Demo Break

In a terminal, from the repo root:

```bash
source .venv/bin/activate
uvicorn backend.main:app --reload
```

Open **http://localhost:8000**, click **Start talking**, and walk through:

1. Create session
2. Connect browser (WebRTC)
3. Capture microphone
4. Send audio
5. Receive audio
6. Play response

Keep DevTools open on the Network/Console tab — the next section is easier to
follow once people have seen the raw events fly by.

## 6. Events Deep Dive

Every state change on the session is one of these events. Point them out live
in DevTools as the conversation runs — students love seeing the raw stream.

In [ ]:
["session.created", "input_audio_buffer.append", "input_audio_buffer.commit", "response.create", "response.audio.delta", "response.audio.done", "response.done"]

## 7. Tool Calling

```
User: "I want to order pizza"
 ↓
GPT → Function Call → FastAPI → JSON → GPT → Voice Response
```

The model never sees SQL, REST endpoints, or internal APIs — only a tool name
and a JSON schema for its arguments. Ours:

- `get_weather(location)`
- `calculate(expression)`
- `get_order_status(order_id)`

Every one of these is dispatched through a single `POST /tools/call` endpoint
on the backend (see `backend/api/tools.py`). With the server running from the
live demo break, we can call it directly, exactly like the browser does:

In [ ]:
import requests
requests.post("http://localhost:8000/tools/call", json={"name": "get_weather", "arguments": {"location": "Pune"}}).json()

## 8. Prompt Engineering for Voice

Chat prompts and voice prompts are not the same thing.

**Instead of:**
```
You are helpful.
```

**Use:**
```
Be concise. Speak naturally.
Never exceed 2 sentences.
Avoid markdown.
Pause before answering.
Confirm actions.
Ask one question at a time.
```

Why: voice output is *heard*, not skimmed. Markdown, long paragraphs, and
bullet lists all sound wrong out loud — see `backend/core/agent-instruction.md`
for the prompt actually driving today's demo.

## 9. Advanced Features (mention only)

- Voice selection
- Interruptions (barge-in)
- Voice Activity Detection (VAD)
- Streaming transcripts
- Emotion
- Background noise handling
- Multi-language support
- Context window management

## 10. Q&A + Next Steps

Where this goes from here:

- MCP integration
- Calendar booking
- RAG
- CRM integration
- Twilio phone agent
- SIP calling
- Browser agent
- Customer support agent

### Closing live demo flow

Back in the browser, run through one exciting end-to-end conversation:

```
"Hi" → AI responds
"What's the weather in Pune?" → calls get_weather → answers
"What's 18 times 24?" → calls calculate → answers
"What's the status of order ORD-1023?" → calls get_order_status → answers
```

One flow, three tool calls, zero typing.